[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/060.%20The%20Warehouse%20Slotting%20Optimization%20Problem/P60-Tier-11_executed.ipynb)

---


# 60. The Warehouse Slotting Optimization Problem

## Tier 11: Physical-Cyber Synthesis

This notebook builds a small, self-contained simulation of a warehouse whose physical state is mirrored by a digital representation. It combines an initial slotting heuristic, sensor-data collection, state synchronization, simple autonomous task assignment, and performance measurement.

### Goals, inputs, and outputs

The simulation creates SKUs, storage locations, robots, and human workers. The initial slotting procedure assigns SKUs in descending order of demand velocity to the most accessible location that has enough remaining space and weight capacity. The simulation then advances in fixed 30-second steps, collects synthetic sensor data, synchronizes the digital state, assigns robot tasks, and records metrics.

The outputs are the generated warehouse state, synchronization and simulation metrics, and a summary of the run. The random seed is fixed so that the generated entities and random decisions can be reproduced, apart from wall-clock timing fields.

### Model assumptions and scope

- This is a synthetic, illustrative simulation. It does not connect to real sensors, robots, workers, or a deployed digital twin.
- Sensor readings and AI decisions are generated by simple random rules; they are not learned from operational data.
- The initial slotting procedure is a greedy heuristic, not a proof of global optimality. It does not model order paths, SKU affinity, replenishment, picking congestion, or service-level constraints.
- Robot tasks are completed immediately in the model. Travel time, collisions, and task interactions are not simulated.
- The simulation does not establish any of the performance-improvement percentages sometimes associated with warehouse technology deployments. Its metrics describe only this model and its assumptions.

### Initial slotting heuristic

A SKU's velocity is a synthetic measure of how frequently it is requested. The heuristic sorts SKUs from highest to lowest velocity and locations from most to least accessible. It assigns each SKU to the first location with enough remaining space and weight capacity.

Let $I$ be the set of SKUs and $L$ the set of locations. For SKU $i$, let $v_i$ be its velocity, $s_i$ its required space, and $w_i$ its weight. For location $\ell$, let $a_\ell$ be its accessibility score, $C_\ell$ its space capacity, and $W_\ell$ its weight limit. Lower accessibility scores mean easier access. The assignment is represented by $x_{i\ell}=1$ when SKU $i$ is assigned to location $\ell$, and $0$ otherwise.

The capacity conditions for an assignment are:

$$\sum_{i\in I} s_i x_{i\ell} \leq C_\ell, \qquad \sum_{i\in I} w_i x_{i\ell} \leq W_\ell \quad (\ell\in L).$$

The code does not solve a mathematical optimization model over all feasible assignments. Instead, it processes SKUs in descending order of $v_i$ and uses the accessibility order as a tie-breaking placement rule. This is the ` _initialize_physical_state` procedure below. Its result is a feasible greedy assignment when capacity permits; a SKU that cannot be placed remains unassigned and is reported in the state.

### Simulation sequence

At each step, the simulation collects synthetic sensor data, updates the physical state, copies that state to its digital counterpart, evaluates synchronization, makes simple agent decisions, executes robot tasks, updates mixed-reality interface records, and stores metrics. The order matters: metrics are calculated after the current step's synchronization and operations.

Synchronization quality is an average of four components: SKU-location agreement, robot-position agreement, human-position agreement, and an assumed location-sensor quality. The average is a model-defined indicator, not a measured real-world accuracy statistic.

The loop uses $T=\lfloor 60D/\Delta t\rfloor$ steps, where $D$ is the duration in minutes and $\Delta t=30$ seconds. The implementation has a fixed step count and does not wait for wall-clock time. Each step processes the entities in the synthetic warehouse, so computational work grows with the number of entities and steps.

In [ ]:
from collections import defaultdict
from datetime import datetime, timedelta
import random
import time
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

np.random.seed(42)
random.seed(42)

In [ ]:
class PhysicalCyberWarehouse:
    """Synthetic warehouse model with physical and digital state."""

    def __init__(
        self,
        num_skus: int = 500,
        num_locations: int = 100,
        num_robots: int = 50,
        num_humans: int = 200,
    ) -> None:
        if min(num_skus, num_locations, num_robots, num_humans) <= 0:
            raise ValueError("All entity counts must be positive.")

        print(
            "Initializing warehouse: "
            f"{num_skus} SKUs, {num_locations} locations, "
            f"{num_robots} robots, and {num_humans} human workers."
        )

        self.skus = self._create_skus(num_skus)
        self.locations = self._create_locations(num_locations)
        self.robots = self._create_robots(num_robots)
        self.humans = self._create_humans(num_humans)
        self.iot_sensors = self._deploy_iot_network()
        self.digital_twin = self._create_digital_twin()
        self.metaverse = self._create_metaverse()
        self.mixed_reality = self._create_mixed_reality()
        self.ai_agents = self._create_ai_agents()
        self.physical_state = self._initialize_physical_state()
        self.cyber_state = {
            "slotting": dict(self.physical_state["slotting"]),
            "location_usage": {
                key: dict(value)
                for key, value in self.physical_state["location_usage"].items()
            },
            "timestamp": datetime.now(),
        }
        self.synchronization_quality = 1.0
        self.performance_metrics = defaultdict(list)

        for sku in self.skus:
            sku["cyber_location"] = sku["physical_location"]

        print("Warehouse model initialized.")

    def _create_skus(self, num_skus: int) -> List[Dict[str, Any]]:
        """Generate SKUs with synthetic demand, space, and weight data."""
        categories = [
            "Electronics", "Clothing", "Food", "Books",
            "Toys", "Sports", "Home", "Beauty",
        ]
        skus = []

        for index in range(num_skus):
            category = str(np.random.choice(categories))
            if category == "Food":
                velocity = np.random.lognormal(4.5, 0.8)
            elif category == "Electronics":
                velocity = np.random.lognormal(3.5, 1.2)
            else:
                velocity = np.random.lognormal(3.0, 1.0)

            skus.append(
                {
                    "id": index + 1,
                    "velocity": max(1.0, float(velocity)),
                    "space_req": float(np.random.uniform(0.1, 4.0)),
                    "weight": float(np.random.uniform(0.1, 50.0)),
                    "category": category,
                    "physical_location": None,
                    "cyber_location": None,
                    "digital_twin_id": f"SKU_{index + 1:06d}",
                }
            )

        return skus

    def _create_locations(self, num_locations: int) -> List[Dict[str, Any]]:
        """Generate storage locations with capacity and accessibility data."""
        zones = ["Fast-Pick", "Reserve", "Cold-Storage", "High-Security", "Bulk"]
        probabilities = [0.3, 0.4, 0.1, 0.1, 0.1]
        grid_size = int(np.sqrt(num_locations)) + 1
        locations = []

        for index in range(num_locations):
            zone = str(np.random.choice(zones, p=probabilities))
            if zone == "Fast-Pick":
                capacity = float(np.random.uniform(8, 20))
                accessibility = float(np.random.uniform(1, 3))
            elif zone == "Reserve":
                capacity = float(np.random.uniform(15, 40))
                accessibility = float(np.random.uniform(4, 8))
            elif zone == "Cold-Storage":
                capacity = float(np.random.uniform(10, 25))
                accessibility = float(np.random.uniform(3, 6))
            else:
                capacity = float(np.random.uniform(20, 50))
                accessibility = float(np.random.uniform(5, 10))

            coordinates = (
                float((index % grid_size) * 5),
                float((index // grid_size) * 5),
                0.0,
            )
            locations.append(
                {
                    "id": index + 1,
                    "capacity": capacity,
                    "weight_limit": capacity * 25,
                    "accessibility": accessibility,
                    "zone": zone,
                    "physical_coordinates": coordinates,
                    "cyber_coordinates": coordinates,
                    "digital_twin_id": f"LOC_{index + 1:06d}",
                    "sensors": [],
                    "current_occupancy": 0.0,
                    "current_weight": 0.0,
                }
            )

        return locations

    def _create_robots(self, num_robots: int) -> List[Dict[str, Any]]:
        """Generate robots and their synthetic operating attributes."""
        robot_types = ["Picker", "Packer", "Transporter", "Inventory", "Maintenance"]
        robots = []

        for index in range(num_robots):
            position = np.random.uniform(0, 100, 3)
            robots.append(
                {
                    "id": f"RBT_{index + 1:04d}",
                    "type": str(np.random.choice(robot_types)),
                    "physical_position": position,
                    "cyber_position": position.copy(),
                    "speed": float(np.random.uniform(1.0, 3.0)),
                    "capacity": float(np.random.uniform(20, 100)),
                    "battery_level": float(np.random.uniform(0.3, 1.0)),
                    "status": "idle",
                    "current_task": None,
                }
            )

        return robots

    def _create_humans(self, num_humans: int) -> List[Dict[str, Any]]:
        """Generate human workers and synthetic efficiency attributes."""
        roles = ["Picker", "Packer", "Supervisor", "Quality Control", "Maintenance"]
        humans = []

        for index in range(num_humans):
            position = np.random.uniform(0, 100, 3)
            humans.append(
                {
                    "id": f"HUM_{index + 1:04d}",
                    "role": str(np.random.choice(roles)),
                    "physical_position": position,
                    "cyber_position": position.copy(),
                    "ar_device": True,
                    "vr_capability": bool(np.random.choice([True, False], p=[0.7, 0.3])),
                    "efficiency_rating": float(np.random.uniform(0.6, 1.0)),
                    "status": "idle",
                    "ar_overlays": [],
                }
            )

        return humans

    def _deploy_iot_network(self) -> Dict[str, List[str]]:
        """Create sensor identifiers for locations, robots, and workers."""
        sensors = {
            "temperature": [], "humidity": [], "motion": [], "weight": [],
            "light": [], "rfid": [], "camera": [], "lidar": [],
            "proximity": [], "battery": [], "vibration": [], "air_quality": [],
        }

        for location in self.locations:
            location_id = location["id"]
            location["sensors"] = [
                f"TEMP_{location_id:03d}", f"HUMID_{location_id:03d}",
                f"MOTION_{location_id:03d}", f"WEIGHT_{location_id:03d}",
                f"LIGHT_{location_id:03d}", f"RFID_{location_id:03d}",
            ]
            sensors["temperature"].append(f"TEMP_{location_id:03d}")
            sensors["humidity"].append(f"HUMID_{location_id:03d}")
            sensors["motion"].append(f"MOTION_{location_id:03d}")
            sensors["weight"].append(f"WEIGHT_{location_id:03d}")
            sensors["light"].append(f"LIGHT_{location_id:03d}")
            sensors["rfid"].append(f"RFID_{location_id:03d}")

        for robot in self.robots:
            sensors["camera"].append(f"CAM_{robot['id']}")
            sensors["lidar"].append(f"LIDAR_{robot['id']}")
            sensors["proximity"].append(f"PROX_{robot['id']}")
            sensors["battery"].append(f"BAT_{robot['id']}")

        for human in self.humans:
            sensors["motion"].append(f"MOTION_{human['id']}")
            sensors["vibration"].append(f"VIB_{human['id']}")
            sensors["air_quality"].append(f"AIR_{human['id']}")

        print(f"Generated {sum(map(len, sensors.values()))} sensor identifiers.")
        return sensors

    def _create_digital_twin(self) -> Dict[str, Any]:
        """Describe the digital representation used by the simulation."""
        zones = {name: [] for name in [
            "Fast-Pick", "Reserve", "Cold-Storage", "High-Security", "Bulk"
        ]}
        for location in self.locations:
            zones[location["zone"]].append(location["id"])
        return {
            "warehouse_model": {
                "dimensions_m": {"length": 100, "width": 80, "height": 15},
                "zones": zones,
                "layout": "grid_based",
            },
            "synchronization": {"sync_frequency_seconds": 1, "latency_seconds": 0.1},
        }

    def _create_metaverse(self) -> Dict[str, Any]:
        """Store the virtual-environment settings represented in the model."""
        return {
            "interaction_mode": "mixed_reality",
            "multiplayer_support": True,
            "simulation_time_scale": 1.0,
        }

    def _create_mixed_reality(self) -> Dict[str, Any]:
        """Store interface capabilities represented in the simulation."""
        return {
            "ar_overlay_types": ["navigation", "information", "training"],
            "vr_use_case": "training_simulation",
        }

    def _create_ai_agents(self) -> List[Dict[str, Any]]:
        """Represent the four agent roles used by the decision process."""
        return [
            {"id": "AI_COORDINATOR", "type": "central_coordinator"},
            {"id": "AI_ROBOT_COORD", "type": "robot_coordinator"},
            {"id": "AI_PREDICTIVE", "type": "predictive_analytics"},
            {"id": "AI_HUMAN_SUPPORT", "type": "human_support"},
        ]

    def _initialize_physical_state(self) -> Dict[str, Any]:
        """Greedily assign high-velocity SKUs to accessible feasible locations."""
        slotting = {}
        location_usage = {
            location["id"]: {"space": 0.0, "weight": 0.0, "count": 0}
            for location in self.locations
        }
        locations_by_access = sorted(
            self.locations, key=lambda location: location["accessibility"]
        )

        for sku in sorted(self.skus, key=lambda item: item["velocity"], reverse=True):
            for location in locations_by_access:
                usage = location_usage[location["id"]]
                has_space = usage["space"] + sku["space_req"] <= location["capacity"]
                has_weight_capacity = (
                    usage["weight"] + sku["weight"] <= location["weight_limit"]
                )
                if has_space and has_weight_capacity:
                    slotting[sku["id"]] = location["id"]
                    sku["physical_location"] = location["id"]
                    usage["space"] += sku["space_req"]
                    usage["weight"] += sku["weight"]
                    usage["count"] += 1
                    break

        for location in self.locations:
            usage = location_usage[location["id"]]
            location["current_occupancy"] = usage["space"]
            location["current_weight"] = usage["weight"]

        capacity_total = sum(location["capacity"] for location in self.locations)
        used_space = sum(usage["space"] for usage in location_usage.values())
        return {
            "slotting": slotting,
            "location_usage": location_usage,
            "timestamp": datetime.now(),
            "total_skus_assigned": len(slotting),
            "unassigned_skus": len(self.skus) - len(slotting),
            "space_utilization": 100 * used_space / capacity_total,
        }

    def synchronize_physical_cyber(self) -> Dict[str, Any]:
        """Collect synthetic readings, mirror state, and measure agreement."""
        start_time = time.perf_counter()
        sensor_data = self._collect_sensor_data()
        self._update_physical_state(sensor_data)
        self._update_cyber_state()
        self._calculate_synchronization_quality()
        elapsed = time.perf_counter() - start_time
        return {
            "sync_time_seconds": elapsed,
            "sync_quality": self.synchronization_quality,
            "sensor_data_groups": len(sensor_data),
            "timestamp": datetime.now(),
        }

    def _collect_sensor_data(self) -> Dict[str, Any]:
        """Generate one synthetic observation set for the current step."""
        return {
            "location_data": {
                location["id"]: {
                    "temperature": float(np.random.uniform(15, 25)),
                    "humidity": float(np.random.uniform(30, 70)),
                    "weight_reading": location["current_weight"],
                    "rfid_tags": sum(
                        sku["physical_location"] == location["id"]
                        for sku in self.skus
                    ),
                }
                for location in self.locations
            },
            "robot_data": {
                robot["id"]: {
                    "position": robot["physical_position"].copy(),
                    "battery_level": robot["battery_level"],
                    "status": robot["status"],
                }
                for robot in self.robots
            },
            "human_data": {
                human["id"]: {
                    "position": human["physical_position"].copy(),
                    "efficiency": human["efficiency_rating"],
                }
                for human in self.humans
            },
            "environmental_data": {
                "ambient_temperature": float(np.random.uniform(18, 22)),
                "ambient_humidity": float(np.random.uniform(40, 60)),
            },
            "timestamp": datetime.now(),
        }

    def _update_physical_state(self, sensor_data: Dict[str, Any]) -> None:
        """Apply readings to the corresponding physical-state records."""
        for location_id, readings in sensor_data["location_data"].items():
            location = self._location_by_id(location_id)
            location["current_weight"] = readings["weight_reading"]
            location["current_occupancy"] = sum(
                self.skus_by_id(sku_id)["space_req"]
                for sku_id, assigned_location in self.physical_state["slotting"].items()
                if assigned_location == location_id
            )

        for robot_id, readings in sensor_data["robot_data"].items():
            robot = next(robot for robot in self.robots if robot["id"] == robot_id)
            robot["physical_position"] = readings["position"]
            robot["battery_level"] = readings["battery_level"]
            robot["status"] = readings["status"]

        for human_id, readings in sensor_data["human_data"].items():
            human = next(human for human in self.humans if human["id"] == human_id)
            human["physical_position"] = readings["position"]
            human["efficiency_rating"] = readings["efficiency"]

        self.physical_state["timestamp"] = sensor_data["timestamp"]

    def _update_cyber_state(self) -> None:
        """Copy physical slotting and usage into the digital state."""
        self.cyber_state = {
            "slotting": dict(self.physical_state["slotting"]),
            "location_usage": {
                key: dict(value)
                for key, value in self.physical_state["location_usage"].items()
            },
            "timestamp": datetime.now(),
        }
        for sku in self.skus:
            sku["cyber_location"] = sku["physical_location"]
        for location in self.locations:
            location["cyber_coordinates"] = location["physical_coordinates"]
        for robot in self.robots:
            robot["cyber_position"] = robot["physical_position"].copy()
        for human in self.humans:
            human["cyber_position"] = human["physical_position"].copy()

    def _calculate_synchronization_quality(self) -> None:
        """Average the modeled SKU, robot, human, and sensor agreement scores."""
        sku_quality = sum(
            sku["physical_location"] == sku["cyber_location"] for sku in self.skus
        ) / len(self.skus)
        robot_quality = sum(
            np.allclose(robot["physical_position"], robot["cyber_position"], atol=0.1)
            for robot in self.robots
        ) / len(self.robots)
        human_quality = sum(
            np.allclose(human["physical_position"], human["cyber_position"], atol=0.1)
            for human in self.humans
        ) / len(self.humans)
        location_sensor_quality = 0.95
        self.synchronization_quality = float(
            np.mean([sku_quality, robot_quality, human_quality, location_sensor_quality])
        )

    def _location_by_id(self, location_id: int) -> Dict[str, Any]:
        return next(location for location in self.locations if location["id"] == location_id)

    def skus_by_id(self, sku_id: int) -> Dict[str, Any]:
        return next(sku for sku in self.skus if sku["id"] == sku_id)

    def _ai_agent_decision_making(self) -> Dict[str, Any]:
        """Generate simple stochastic slotting and robot-task decisions."""
        decisions = {"slotting_optimizations": [], "robot_tasks": []}

        if len(self.skus) >= 5 and random.random() < 0.3:
            decisions["slotting_optimizations"].append(
                {
                    "target_skus": random.sample(self.skus, 5),
                    "reason": "synthetic_demand_pattern_change",
                }
            )

        idle_robots = [robot for robot in self.robots if robot["status"] == "idle"]
        for robot in idle_robots[:10]:
            decisions["robot_tasks"].append(
                {
                    "robot_id": robot["id"],
                    "target_location": random.choice(self.locations)["id"],
                    "estimated_duration": float(np.random.uniform(5, 30)),
                }
            )

        return decisions

    def _execute_autonomous_operations(
        self, decisions: Dict[str, Any]
    ) -> Dict[str, Any]:
        """Apply the decisions and report completed modeled operations."""
        applied = []
        for optimization in decisions["slotting_optimizations"]:
            if self._apply_slotting_optimization(optimization):
                applied.append(optimization)

        completed_tasks = []
        for task in decisions["robot_tasks"]:
            if self._execute_robot_task(task):
                completed_tasks.append(task)

        return {"optimizations_applied": applied, "completed_tasks": completed_tasks}

    def _apply_slotting_optimization(self, optimization: Dict[str, Any]) -> bool:
        """Move selected SKUs only when accessibility improves and capacity allows."""
        all_moves_succeeded = True
        for sku in optimization["target_skus"]:
            current_id = sku["physical_location"]
            if current_id is None:
                all_moves_succeeded = False
                continue

            current_location = self._location_by_id(current_id)
            candidates = []
            for location in self.locations:
                if location["id"] == current_id:
                    continue
                if location["accessibility"] >= current_location["accessibility"]:
                    continue
                usage = self.physical_state["location_usage"][location["id"]]
                if usage["space"] + sku["space_req"] > location["capacity"]:
                    continue
                if usage["weight"] + sku["weight"] > location["weight_limit"]:
                    continue
                if usage["count"] >= 5:
                    continue
                candidates.append(location)

            if not candidates:
                all_moves_succeeded = False
                continue

            new_location = min(candidates, key=lambda item: item["accessibility"])
            old_usage = self.physical_state["location_usage"][current_id]
            new_usage = self.physical_state["location_usage"][new_location["id"]]
            old_usage["space"] -= sku["space_req"]
            old_usage["weight"] -= sku["weight"]
            old_usage["count"] -= 1
            new_usage["space"] += sku["space_req"]
            new_usage["weight"] += sku["weight"]
            new_usage["count"] += 1
            sku["physical_location"] = new_location["id"]
            sku["cyber_location"] = new_location["id"]
            self.physical_state["slotting"][sku["id"]] = new_location["id"]

        for location in self.locations:
            usage = self.physical_state["location_usage"][location["id"]]
            location["current_occupancy"] = usage["space"]
            location["current_weight"] = usage["weight"]
        return all_moves_succeeded

    def _execute_robot_task(self, task: Dict[str, Any]) -> bool:
        """Simulate an immediate robot task and its battery use."""
        robot = next(
            (item for item in self.robots if item["id"] == task["robot_id"]), None
        )
        if robot is None:
            return False

        robot["status"] = "working"
        robot["current_task"] = task
        consumption = task["estimated_duration"] * 0.01
        robot["battery_level"] = max(0.1, robot["battery_level"] - consumption)
        target = self._location_by_id(task["target_location"])
        robot["physical_position"] = np.asarray(target["physical_coordinates"])
        robot["cyber_position"] = robot["physical_position"].copy()
        robot["status"] = "idle"
        robot["current_task"] = None
        return True

    def _update_mixed_reality(self) -> Dict[str, Any]:
        """Create simple AR guidance and record synthetic VR participation."""
        ar_users = [human for human in self.humans if human["ar_device"]]
        vr_users = [human for human in self.humans if human["vr_capability"]]
        overlays = []
        for human in ar_users:
            human["ar_overlays"] = [
                {"type": "navigation", "content": "Assigned warehouse task"},
                {
                    "type": "information",
                    "content": f"Efficiency rating: {human['efficiency_rating']:.1%}",
                },
            ]
            overlays.extend(human["ar_overlays"])
        vr_sessions = sum(random.random() < 0.3 for _ in vr_users)
        return {
            "ar_users": len(ar_users),
            "vr_capable_users": len(vr_users),
            "vr_sessions": vr_sessions,
            "overlays": overlays,
        }

    def _calculate_simulation_metrics(self) -> Dict[str, float]:
        """Calculate model metrics using explicitly defined denominators."""
        total_capacity = sum(location["capacity"] for location in self.locations)
        used_space = sum(
            usage["space"]
            for usage in self.physical_state["location_usage"].values()
        )
        active_robots = sum(robot["status"] != "idle" for robot in self.robots)
        average_human_efficiency = float(
            np.mean([human["efficiency_rating"] for human in self.humans])
        )
        return {
            "picking_efficiency_percent": average_human_efficiency * 100,
            "space_utilization_percent": 100 * used_space / total_capacity,
            "robot_utilization_percent": 100 * active_robots / len(self.robots),
            "human_efficiency_percent": average_human_efficiency * 100,
            "sync_quality_percent": self.synchronization_quality * 100,
            "throughput_orders_per_hour": 0.0,
            "ar_adoption_percent": 100 * sum(
                human["ar_device"] for human in self.humans
            ) / len(self.humans),
            "vr_capability_percent": 100 * sum(
                human["vr_capability"] for human in self.humans
            ) / len(self.humans),
        }

    def run_metaverse_simulation(self, duration_minutes: int = 60) -> Dict[str, Any]:
        """Run fixed-step simulation and return metric histories and averages."""
        if duration_minutes <= 0:
            raise ValueError("duration_minutes must be positive.")

        print(f"Running simulation for {duration_minutes} simulated minutes.")
        start_time = time.perf_counter()
        step_seconds = 30
        total_steps = duration_minutes * 60 // step_seconds
        metrics_history = defaultdict(list)
        timestamps = []
        current_time = datetime.now()

        for step in range(total_steps):
            self.synchronize_physical_cyber()
            decisions = self._ai_agent_decision_making()
            self._execute_autonomous_operations(decisions)
            self._update_mixed_reality()
            step_metrics = self._calculate_simulation_metrics()
            for name, value in step_metrics.items():
                metrics_history[name].append(value)
            timestamps.append(current_time)
            current_time += timedelta(seconds=step_seconds)

        elapsed = time.perf_counter() - start_time
        averages = {
            f"average_{name}": float(np.mean(values))
            for name, values in metrics_history.items()
        }
        results = {
            "simulation_time_seconds": elapsed,
            "total_steps": total_steps,
            "metrics_history": dict(metrics_history),
            "timestamps": timestamps,
            **averages,
        }
        print(f"Completed {total_steps} simulation steps.")
        print(f"Average synchronization quality: {averages['average_sync_quality_percent']:.2f}%")
        print(f"Average space utilization: {averages['average_space_utilization_percent']:.2f}%")
        return results

    def generate_performance_summary(
        self, simulation_results: Dict[str, Any]
    ) -> Dict[str, Any]:
        """Collect scale, assignment, and simulation statistics."""
        total_sensor_ids = sum(len(items) for items in self.iot_sensors.values())
        summary = {
            "warehouse_scale": {
                "skus": len(self.skus),
                "assigned_skus": self.physical_state["total_skus_assigned"],
                "unassigned_skus": self.physical_state["unassigned_skus"],
                "locations": len(self.locations),
                "robots": len(self.robots),
                "humans": len(self.humans),
                "sensor_identifiers": total_sensor_ids,
            },
            "simulation": {
                "total_steps": simulation_results["total_steps"],
                "wall_clock_seconds": simulation_results["simulation_time_seconds"],
                "average_sync_quality_percent": simulation_results[
                    "average_sync_quality_percent"
                ],
                "average_space_utilization_percent": simulation_results[
                    "average_space_utilization_percent"
                ],
                "average_robot_utilization_percent": simulation_results[
                    "average_robot_utilization_percent"
                ],
                "average_human_efficiency_percent": simulation_results[
                    "average_human_efficiency_percent"
                ],
            },
        }
        return summary

## Run the warehouse simulation

The instance below uses the example scale of 300 SKUs, 50 locations, 30 robots, and 100 human workers, and simulates 30 minutes in 30-second steps. These are model inputs, not measured warehouse operating data. The notebook reports values calculated by the code when the cells are run; it does not embed precomputed performance claims.

In [ ]:
metaverse_warehouse = PhysicalCyberWarehouse(
    num_skus=300,
    num_locations=50,
    num_robots=30,
    num_humans=100,
)

initial_sync_results = metaverse_warehouse.synchronize_physical_cyber()
simulation_results = metaverse_warehouse.run_metaverse_simulation(
    duration_minutes=30
)
performance_summary = metaverse_warehouse.generate_performance_summary(
    simulation_results
)

print("\nWarehouse scale and assignment:")
for name, value in performance_summary["warehouse_scale"].items():
    print(f"  {name.replace('_', ' ').title()}: {value}")

print("\nSimulation averages:")
for name, value in performance_summary["simulation"].items():
    print(f"  {name.replace('_', ' ').title()}: {value:.3f}")

## Results interpretation

The generated summary separates warehouse scale and assignment information from simulation metrics. The assigned and unassigned SKU counts show how the greedy procedure behaved under the generated capacities. A nonzero unassigned count means that the procedure could not place every SKU while respecting its space and weight checks; it does not imply that no feasible assignment exists, because the procedure is not an exhaustive search.

Space utilization is the total space assigned to SKUs divided by the sum of location capacities. Robot utilization is measured at the instant metrics are calculated. Because modeled robot tasks begin and finish within one step, this instantaneous measure can be zero even though tasks were completed during the run. The throughput metric is explicitly set to zero because the simulation does not model orders or order completion. Human efficiency is the average of the generated worker-efficiency ratings, not a measured productivity rate.

Synchronization quality is a model indicator whose SKU and worker/robot components compare physical and digital records after copying the physical state. Its fixed location-sensor component is an assumption. It should therefore be interpreted as a consistency check within the simulation, not as a validated real-world accuracy percentage.

The simulation illustrates a sequence of physical-state updates, digital-state mirroring, simple agent decisions, and metric calculation. It does not demonstrate that a metaverse, autonomous coordination system, or slotting policy produces a particular operational improvement. Such conclusions would require validated data, explicit operational objectives, and an evaluation design beyond this synthetic example.

### Further study

Within this modeling framework, a natural extension is to examine how the existing heuristic's assignments respond to different SKU mixes, location capacities, or accessibility scores. Any such experiment should keep the assumptions and random seed explicit and distinguish simulated outcomes from empirical warehouse performance.